<a href="https://colab.research.google.com/github/Sian118/LabAssignment/blob/main/Lecture_17_Data_Cleaning_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
import pandas as pd
# STEP 1: Upload and load the CSV
from google.colab import files
uploaded = files.upload()

print("The pandas have been imported and their version is", pd.__version__)

Saving messy_orders.csv to messy_orders (2).csv
The pandas have been imported and their version is 2.2.3


In [51]:
df=pd.read_csv("messy_orders.csv")
df
 #STEP 2: Keep a raw, untouched copy
# for comparison later
raw_df=df.copy()


In [52]:
# STEP 3: Inspect the raw data
# before changing anything
print("shape",df.shape)
print("Column",df.columns)
df.info()
print(df.isna().sum())
print("Duplicated rows:",df.duplicated().sum())


shape (10, 6)
Column Index(['Order_ID', 'Customer', 'City', 'Qty', 'Price', 'Order_Date'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Order_ID    10 non-null     int64 
 1   Customer    10 non-null     object
 2   City        10 non-null     object
 3   Qty         10 non-null     object
 4   Price       10 non-null     object
 5   Order_Date  10 non-null     object
dtypes: int64(1), object(5)
memory usage: 612.0+ bytes
Order_ID      0
Customer      0
City          0
Qty           0
Price         0
Order_Date    0
dtype: int64
Duplicated rows: 1


In [53]:
# City is missing for Order 1004. It's a categorical column, and I have
# no evidence of the real value, so I'm labeling it "Unknown" instead of guessing
df["City"]=df["City"].replace('-',pd.NA).fillna("Unknown")

In [54]:
 #Price is missing for Order 1007. First I need to remove the "$" and ","
# formatting and convert to numeric before I can calculate anything.
pd_numeric=pd.to_numeric(df["Price"].replace("-",pd.NA).astype("str")
.str.replace("$","",regex=False)
.str.replace(",","",regex=False),
 errors="coerce",


)
df["Price"]=pd_numeric.fillna(pd_numeric.median())


In [55]:
# keep=False shows BOTH copies of any duplicated row, so I can visually
duplicates=df[df.duplicated(keep=False)]
print(duplicates)

   Order_ID   Customer     City Qty   Price  Order_Date
5      1006  Sara Khan  Karachi   2  2500.0  2026-09-03
6      1006  Sara Khan  Karachi   2  2500.0  2026-09-03


In [56]:
# Confirmed: Order 1006 is an exact duplicate. Safe to remove.
df=df.drop_duplicates().copy()

In [57]:
# "lahore", "LAHORE", and "Lahore" all mean the same city, but Pandas
# treats them as 3 different categories unless I standardize the text.
df["City"]=df["City"].str.strip().str.title()


In [58]:
# Qty mixes real numbers with the word "two". pd.to_numeric() doesn't
# understand English words, so I manually replace "two" with "2" first
df["Qty"]=df["Qty"].replace('Two',"2")
df["Qty"]=pd.to_numeric(df["Qty"],errors='coerce')

In [59]:
# Order_Date is written in several different formats. format="mixed" tells
# Pandas to inspect each value individually and figure out its own format,
# instead of assuming one fixed pattern for the whole column.
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"], format="mixed", dayfirst=True, errors="coerce"
)

In [60]:
#Sort by price descending to surface the most suspicious (highest) values first.
print(df.sort_values("Price", ascending=False)[["Order_ID", "Customer", "Price"]].head())

   Order_ID     Customer     Price
4      1005  Bilal Ahmed  450000.0
2      1003  Hamza Iqbal    4500.0
8      1008    Usman Ali    3800.0
9      1009    Hira Noor    3200.0
1      1002    Sara Khan    3200.0


In [61]:
# Qty should be 1, not -1. Price should be 4,500, not 450,000.
# I only correct this because it's confirmed - not just because it looked unusual.
df.loc[df["Order_ID"]==1005,"Qty"]=1
df.loc[df["Order_ID"]==1005,"Price"]=4500

In [62]:
# Qty and Price are now both clean numeric columns, so this calculation
# is reliable. I only create this AFTER cleaning, not before.
df["Order_Value"]=df["Qty"]*df["Price"]


In [63]:
# Re-running the SAME checks from Step 3 to confirm the fixes actually worked.
print(df.shape)
print(df.isna().sum())
print("Duplicates:", df.duplicated().sum())
df.info()

(9, 7)
Order_ID       0
Customer       0
City           0
Qty            1
Price          0
Order_Date     0
Order_Value    1
dtype: int64
Duplicates: 0
<class 'pandas.core.frame.DataFrame'>
Index: 9 entries, 0 to 9
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Order_ID     9 non-null      int64         
 1   Customer     9 non-null      object        
 2   City         9 non-null      object        
 3   Qty          8 non-null      float64       
 4   Price        9 non-null      float64       
 5   Order_Date   9 non-null      datetime64[ns]
 6   Order_Value  8 non-null      float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(2)
memory usage: 576.0+ bytes


In [64]:
display(raw_df)
display(df)

,Order_ID,Customer,City,Qty,Price,Order_Date
0,1001,Ali Khan,Lahore,2,"$2,500",2026-09-01
1,1002,Sara Khan,lahore,1,"$3,200",01/09/2026
2,1003,Hamza Iqbal,Karachi,1,"$4,500",2026/09/02
3,1004,Ayesha Noor,—,3,"$1,800",02-Sep-2026
4,1005,Bilal Ahmed,LAHORE,-1,"$450,000",2026-09-03
5,1006,Sara Khan,Karachi,2,"$2,500",2026-09-03
6,1006,Sara Khan,Karachi,2,"$2,500",2026-09-03
7,1007,Zoya Malik,Islamabad,1,—,2026-09-04
8,1008,Usman Ali,Karachi,two,"$3,800",2026-09-04
9,1009,Hira Noor,lahore,1,"$3,200",2026-09-05


,Order_ID,Customer,City,Qty,Price,Order_Date,Order_Value
0,1001,Ali Khan,Lahore,2.0,2500.0,2026-09-01,5000.0
1,1002,Sara Khan,Lahore,1.0,3200.0,2026-09-01,3200.0
2,1003,Hamza Iqbal,Karachi,1.0,4500.0,2026-09-02,4500.0
3,1004,Ayesha Noor,—,3.0,1800.0,2026-09-02,5400.0
4,1005,Bilal Ahmed,Lahore,1.0,4500.0,2026-09-03,4500.0
5,1006,Sara Khan,Karachi,2.0,2500.0,2026-09-03,5000.0
7,1007,Zoya Malik,Islamabad,1.0,3200.0,2026-09-04,3200.0
8,1008,Usman Ali,Karachi,NaN,3800.0,2026-09-04,NaN
9,1009,Hira Noor,Lahore,1.0,3200.0,2026-09-05,3200.0
